# သင်ခန်းစာ ၁၃ - အေးဂျင့်မှတ်ဉာဏ်


## ตั้งค่า

โน้ตบุ๊กนี้สาธิตวิธีการสร้างตัวแทนจองการเดินทางที่มี **หน่วยความจำถาวร** โดยใช้ **Microsoft Agent Framework** (MAF)။

คุณจะได้เรียนรู้ว่าสมรรถภาพหน่วยความจำของตัวแทนในรูปแบบต่างๆ — การทำงาน, ระยะสั้น, และระยะยาว — มีผลต่อวิธีที่ตัวแทนเก็บรักษาและใช้ข้อมูลระหว่างการสนทนาอย่างไร

**ข้อกำหนดเบื้องต้น:**
- โครงการ Microsoft Foundry ที่มีโมเดลแชทที่ปรับใช้แล้ว (เช่น `gpt-4.1-mini`)
- ลงชื่อเข้าใช้ด้วย Azure CLI — รัน `az login` ในเทอร์มินัลของคุณ
- `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดโครงการ Microsoft Foundry ของคุณ
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณได้ปรับใช้


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Agent မှတ်ဉာဏ်အမျိုးအစားများ

AI agent များသည် အသုံးပြုနိုင်သော မှတ်ဉာဏ်အမျိုးအစားများ ရှိပြီး၊ အစဉ်အဆက်အသီးသီး၏ ရည်ရွယ်ချက်ကို ပြုလုပ်ပေးသည်။

### အလုပ်လုပ်မှု မှတ်ဉာဏ်
စကားပြောဆိုမှုဆက်စပ်မှု - တစ်ကြိမ်သော ဆွေးနွေးပွဲအတွင်း လဲလှယ်ခဲ့သော မက်ဆေ့ချ်များ။ Agent သည် coherence ထိန်းသိမ်းရန် အဲဒီ ဆွေးနွေးခြင်းထဲမှာ အရင်က မက်ဆေ့ခ််တွေကို ပြန်ပြောကြားနိုင်သည်။ MAF အတွင်းတွင် **`agent.create_session()`** ဖြင့် ဖန်တီးပြီး၊ `AgentSession` ကို return ပြန်ပေးသည်။

### သော့ကြပ် မှတ်ဉာဏ်
တာဝန် သို့မဟုတ် ဆွေးနွေးပွဲ သက်တမ်းအတွင်း တည်တံ့နေသော သတင်းအချက်အလက်များ၊ သိုလှောင်မထားပါ။ ဥပမာအားဖြင့်၊ agent သည် 多回数 စီစဉ်ဆွေးနွေးမှုအတွင်း ထင်မြင်ချက်များ စုဆောင်းပြီး နောက်ဆုံး အစီအစဉ်တစ်ခု ပြုလုပ်ရန် အသုံးပြုနိုင်သည်။

### ရေရှည် မှတ်ဉာဏ်
ဆက်တိုက်ဆာဗာများတစ်လျှောက် ** ထပ်တလဲလဲ ဖြစ်နေသော ** သဘောထားများနှင့် သတင်းအချက်များ။ ပြန်လာသော အသုံးပြုသူသည် သူ၏ စားသောက် ကန့်သတ်ချက်များ သို့မဟုတ် ခရီးသွားပုံစံကို ချို့သောမလုပ်ရန် လိုမည်မဟုတ်ပါ။ ရေရှည် မှတ်ဉာဏ်သည် တခြား အရင်းအမြစ်တစ်ခုဖြစ်သည့် ဒေတာဘေ့စ်၊ ဖိုင် သို့ vector index တို့ဖြင့် ထောက်ပံ့ထားပြီး tools များမှတဆင့် agent သို့ ဖော်ပြသည်။


## ဆက်သွယ်မှုအတောင်းအဖြေမှတ်ဉာဏ်နှင့်အလုပ်လုပ်စနစ်

မှတ်ဉာဏ်၏လွယ်ကူဆုံးပုံစံမှာ အပြောအဆိုဆက်သွယ်မှုအဆက်ဖြစ်သည်။ အတူတူသော session (`agent.create_session()` ဖြင့်ဖန်တီးထားသော) ကို ဆက်တိုက် `agent.run()` ခေါ်သုံးသောအခါ အေဂျင့်သည် အဲဒီအပြောအဆို၏ အပြည့်အစုံမှတ်တမ်းကိုမြင်နိုင်ပြီး ယခင်အချက်အလက်များကိုမှတ်ပုံတင်နိုင်သည်။

ခရီးသွား အေဂျင့်တစ်ဦးဖန်တီးပြီး အလုပ်လုပ်သည့် မှတ်ဉာဏ်ကိုပြသကြမယ်။


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ကိုယ်စားလှယ်သည် အတိအကျ အကြိုပေးရေးကဏ္ဍကို မှတ်မိခဲ့သည်မှာ မက်ဆေ့ချ်နှစ်ခုစလုံးသည် တစ်ချက်တည်းသော အစည်းအဝေးကို မျှဝေပေးနေသည့်အတွက် ဖြစ်သည်။ ဤသည်မှာ **အလုပ်လုပ်နေသော စွမ်းဆောင်ရည်** ဖြစ်ပြီး — ၎င်းသည် အစည်းအဝေး၏ အသက်တမ်းအတွင်းသာ တည်ရှိသည်။

### စာတန်းသစ်တစ်ခုဖြစ်ပါကဘာတွေဖြစ်မလဲ?

ကျွန်ုပ်တို့သည် **အသစ်** အစည်းအဝေးတစ်ခုကို ဖန်တီးပါက၊ ကိုယ်စားလှယ်၌နောက်ဆုံးစကားပွဲအကြောင်း မှတ်မိမှုမရှိပါ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ရေရှည်မှတ်ဉာဏ်ပုံစံ

အသုံးပြုသူနှစ်သက်မှုများကို **အစည်းအဝေးတစ်ခုနှင့်တစ်ခုအတွင်း မဟုတ်ပဲ** မှတ်မိထားရန်အတွက် စကားပြောငြိမ်းချမ်းမှုလမ်းကြောင်းပြင်ပမှာရှိသော အမြဲတမ်းတည်တံ့သောသိုလှောင်မှုတစ်ခုလိုအပ်သည်။ Agent သည် **ကိရိယာများ** မှတဆင့် ဒီသိုလှောင်မှုကို ရယူသို့မဟုတ် သိမ်းဆည်းနိုင်သည်။

အောက်တွင်မှာ ဤကိုရိုးရှင်းသည့် အတွင်းမှတ်ဉာဏ်နှစ်သက်မှုသိုလှောင်မှု ကို ဆောင်ရွက်ထားပြီး (ထုတ်လုပ်မှုအတွက် သင်သည် ဒေတာဘေ့စ် သို့မဟုတ် vector index နဲ့ ထောက်ပံ့ရပါမည်) agent သုံးစွဲနိုင်သော ကိရိယာများအဖြစ် ဖော်ပြထားပါသည်။

### ဖွဲ့စည်းပုံ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ရှေ့ဆွဲ ၁ — ပထမဆုံးအသုံးပြုသူ အနှစ်ပြည့်ခရီးစဉ်စာရင်းသွင်းခြင်း

ဆာရာ ပထမဆုံးလည်ပတ်လာသည်။ ကိုယ်စားလှယ်သည် သူမ၏ အကြိုက်များကို ကိရိယာများဖြင့် သိမ်းဆည်းပြီး ဟိုတယ်များကို အကြံပြုရန် အသုံးပြုသင့်သည်။


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### အကြောင်းအရာ ၂ — ဆာရာအပတ်ပေါင်းများကြာပြီး ပြန်လာသည်

ဆာရာသည် **အလွန်အသစ်သော ခေါင်းစဉ်** တစ်ခုကို စတင်သည် (အစုံအသစ် တစ်ခုကို မိမိလိုချင်သလို သက်မှတ်ခြင်း။) အလုပ်သိမ်းမှတ်ဉာဏ်အတွင်းမှာ အလွတ်ရှိပေမယ့် ရေရှည်နှစ်သက်မှု သိမ်းဆည်းရာတွင် သူမအချက်အလက်များက ရှိနေပါသည်။ ကုန်ကြမ်းသည် ထိုအချက်အလက်များကို ပြန်ယူပြီး ကိုယ်ပိုင် အကြံပြုချက်များတွင် အသုံးပြုသင့်သည်။


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## အနှစ်ချုပ်

ဤသင်ခန်းစာတွင် သင်သည် agent သုံးမျိုးသောမှတ်ဉာဏ်အမျိုးအစားများနှင့် မိုက်ခရိုဆော့ဖ် Agent Framework ဖြင့် ပြုလုပ်ပုံကို လေ့လာခဲ့သည်။

| မှတ်ဉာဏ်အမျိုးအစား | MAF မက်ခနစ် | အသက်တာကာလ |
|---|---|---|
| **အလုပ်လုပ်ဆောင်နေသည့်** | `agent.create_session()` | တစ်စကားဝိုင်းတည်း |
| **ကာလတို** | ကြောင်းအတွင်း စုစည်းထားသော စာရင်းအင်း | တစ်လုပ်ငန်း / အစည်းအဝေးတည်း |
| **ကာလရှည်** | `@tool` ဖန်ရှင်များမှတဆင့် လက်လှမ်းမီသော အပြင်ဘက်သိုလှောင်မှု | အစည်းအဝေးများအနှံ့ |

### အဓိကယူဆချက်များ
1. **`agent.create_session()`** သည် အလုပ်လုပ်ဆောင်နေသော မှတ်ဉာဏ်ကို ပေးသည် — agent သည် session အတွင်း စကားဝိုင်းအပြည့်အစုံကိုမြင်နိုင်သည်။
2. **အစည်းအဝေးအသစ်များသည် စာရင်းအင်းကိုဆုံးရှုံးသည်** — ကာလရှည် မှတ်ဉာဏ်မရှိပါက agent သည် ယခင်စကားဝိုင်းများကို နောက်ဆုတ်မရနိုင်ပါ။
3. **`@tool` ဖန်ရှင်များ** သည် ကြားခံကူးတိုင်များဖြစ်သည် — agent သည် သိုလှောင်ထားသော မဟာဗျူဟာမှ သတင်းအချက်အလက် ထည့်သိမ်းခြင်းနှင့် ရယူခြင်း ပြုလုပ်နိုင်သည်။
4. **ကိုယ်ပိုင်မှုသည် အချိန်ကြာလာတာနဲ့ တိုးတက်လာသည်** — သိုလှောင်ထားသော ဦးစားပေးချက်များ ပိုမိုများလာသည်နှင့်အမျှ agent ၏ အကြံပြုချက်များ ပိုမိုကောင်းမွန်လာသည်။

### လက်တွေ့အသုံးချမှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု**: ဖောက်သည်ဆိုင်ရာ သမိုင်းနှင့် သဘောတူချက်များကို မှတ်မိထားသည်
- **ကိုယ်ပိုင် အကူအညီပေးသူများ**: ရက်ပေါင်းများစွာ သို့မဟုတ် အပတ်ပေါင်းများစွာ အတွင်း အခြေအနေကို ထိန်းသိမ်းထားသည်
- **ကျန်းမာရေးစောင့်ရှောက်မှု**: လူနာဆိုင်ရာသတင်းအချက်အလက်နှင့် သဘောတူချက်များကို လိုက်လံစောင့်ကြည့်သည်
- **အွန်လိုင်းကဏ္ဍခြင်း**: သမိုင်းအခြေပြု ကိုယ်ပိုင် ဈေးဝယ်ခြင်းများ

### နောက်တစ်ဆင့်များ
- မှတ်ဉာဏ်ဒစ်ရှင်နာရီကို ဒေတာဘေ့စ် သို့မဟုတ် vector store (ဥပမာ Azure AI Search) ဖြင့် အစားထိုးသည်
- အချိန်ပတ်သက်သည့် သတင်းအချက်အလက်များအတွက် မှတ်ဉာဏ်သက်တမ်းကုန်ရှိဆုံးခြင်း ထည့်သွင်းသည်
- ဖြန့်ဝေမှတ်ဉာဏ်နှင့် multi-agent စနစ်များ တည်ဆောက်သည်
- Cognee notebook ကို အသုံးပြု၍ သိမြင်မှု-ဂရပ် ဖော်ဆောင်ထားသော မှတ်ဉာဏ်ကို ရှာဖွေသည်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
